# Analysis of gene expression, cbdb

In [ ]:
# run in py_decoupler

In [ ]:
import matplotlib.pyplot as plt
import tqdm as notebook_tqdm
from datetime import date
import seaborn as sns
import pandas as pd
import scanpy as sc
import numpy as np
import scipy
import sys
import os
import re
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
#import decoupler as dc
import decoupler as dc, omnipath as op


#import src.visualization.scBasic as kvis

f"Last execution: {date.today()}"

In [ ]:
from matplotlib import rcParams

In [ ]:
print(dc.__version__) 
print(np.__version__)
print(sc.__version__)
print(sys.version)

## Set input and output directories

In [ ]:
# -- Set base directory
base_dir = '/nfs/team292/rs40/projects/Aneuploid_screen_v2'
sys.path.insert(1, base_dir)
os.chdir(base_dir)

pd.set_option("display.max_columns", 50)
%matplotlib inline

In [ ]:
#meta file 
#meta = '/nfs/team292/rs40/projects/Aneuploid_screen/processed_data/meta_integrated.csv'

In [ ]:
# -- Outputs
#datafiles
output_dir = base_dir+'/processed_data/11_geneexpression'
path = os.path.join(base_dir, output_dir)

if not os.path.exists(path):
    os.makedirs(path)

#figures
figure_dir = output_dir


## Default scanpy settings

In [ ]:
sc.settings.figdir = path

plt.rcParams.update({
    "figure.figsize": (3, 3),    # Default figure size
    "figure.dpi": 300,           # High resolution
    "font.size": 7,              # Global font size (fallback)
    "axes.titlesize": 7,        # Title size (e.g., 'Leiden')
    "axes.labelsize": 7,         # Axis labels (e.g., 'UMAP1')
    "xtick.labelsize": 7,        # Tick numbers on X axis & Colorbars
    "ytick.labelsize": 7,        # Tick numbers on Y axis & Colorbars
    "legend.fontsize": 7,        # Legend text
    "lines.markersize": 1,       # Dot size in legends
    "axes.spines.top": False,    # Remove top border globally
    "axes.spines.right": False   # Remove right border globally
})


sc.settings.figdir = output_dir

sc.set_figure_params(
    scanpy=True,           # Use Scanpy's opinionated style defaults
    dpi=300,               # High resolution for publication
    dpi_save=300,          # Resolution for saved files
    frameon=True,         # Remove box around plots (cleaner)
    vector_friendly=False,  # usage for PDF/SVG editors (Illustrator)
    fontsize=7,            # Set the base font size (very small for 3-inch figures)
    figsize=(3, 3),        # Set default figure size
    #facecolor=None,     # Ensure background is white (not transparent)
    format='pdf'           # Default save format
)

plt.rcParams['xtick.labelsize'] = 7
plt.rcParams['ytick.labelsize'] = 7

In [ ]:
celltype_colors = {
    "early EPI": "#E14A5B",
    "Prelineage": "#FBD8EC",  # Assuming '#premorula' corresponds to '#FBD8EC'
    "Morula": "#D74426",
    "late EPI": "#AE3B36",
    "premorula": "#FBD8EC",
    "ECTO": "#FF80A9",
    "EPI.PrE.INT": "#E1529D",
    "AdvMes": "#AB4979",
    "Mesoderm": "#E1529D",
    "ExE_Mes": "#D82679",
    "PriS": "#A777CD",
    "YSE": "#690FA0",
    "Hypoblast": "#7B36FF",
    "early TE": "#B3D51C",
    "TE": "#47A79B",
    "late TE": "#49B91B",
    "polar TE": "#A79E33",
    "CTB": "#418A7D",
    "STB": "#508E36",
    "EVT": "#849D2C",
    "DE": "#00AAD6",
    "doublet": "#BBBBBB",
    "unknown": "#676767"
}


celltype_colors = {
    "naive EPI":    "#ecbb5f",  # warm yellow
    "blastoid EPI": "#f69320",  # orange
    "TE":       "#4f5b9e",  # blue
    "unspecified":       "#676868",  
}


In [ ]:
custom_palette = [
  "#D3D3D4","#555e7b", "#b7d968", "#b576ad", "#e04644", "#fde47f", "#7ccce5", 
  "#C6E5D9", "#F0A830", "#e04644", "#C0D860", "#F2F26F", "#A8E6CE", 
  "#CCC68D", "#EB6841", "#E1F5C4", "#D9CEB2", "#C5CEAE", "#E84A5F", 
  "#A0C55F", "#DCE9BE", "#FFAAA6", "#F07818", "#E08E79", "#A0C55F", 
  "#948C75", "#C0D860", "#005F6B", "#45484B", "#0B2E59", "#FFF7BD", 
  "#CFBE27", "#F1D4AF", "#C02942", "#5E412F", "#355C7D", "#F27435", 
  "#AAB3AB", "#4ECDC4", "#8C2318", "#FF9E9D", "#E6AC27", "#C7F464", 
  "#4ECDC4", "#ED303C", "#F4FAD2", "#F07818", "#031634", "#838689", 
  "#73626E", "#F9D423", "#C06C84", "#F04155", "#F5634A", "#DFBA69", 
  "#A0C55F"
]

In [ ]:
aneu_col = { "monosomy": "#109E9D",
        "trisomy": "#F26B3B",
        'complex':"#662C91", 
        "diploid": "#C5C5C5"}

In [ ]:
white_to_magenta = LinearSegmentedColormap.from_list(
    "white_to_891753", ["#FFFFFF", "#891753"]
)

white_to_cyan = LinearSegmentedColormap.from_list(
    "white_to_cyan", ["#FFFFFF", "#148991"]
)

## Load data

In [ ]:
adata = sc.read_h5ad("/nfs/team292/rs40/projects/Aneuploid_screen_v2/processed_data/3_scanpy_integration/integrated_sub_adata.h5ad")

In [ ]:
adata.obs

In [ ]:
adata.obs.columns

In [ ]:
adata

In [ ]:
from scipy.io import mmwrite

# 1. Subset the adata to only highly variable genes
# This checks the 'highly_variable' column in adata.var
#adata_hvg = adata[:, adata.var['highly_variable']].copy()

adata_hvg = adata

print(f"Original shape: {adata.shape}")
print(f"HVG subset shape: {adata_hvg.shape}")

# 2. Save the Metadata (obs) - use the same barcodes
adata_hvg.obs.to_csv(output_dir + "/metadata.csv")

# 3. Save the HVG Gene Names (var)
# These will be the top informative genes
pd.DataFrame(adata_hvg.var_names).to_csv(output_dir + "/genes_hvg.csv", index=False)

# 4. Save the UMAP coordinates
umap_df = pd.DataFrame(adata_hvg.obsm['X_umap'], index=adata_hvg.obs_names, columns=['UMAP1', 'UMAP2'])
umap_df.to_csv(output_dir + "/umap_coords.csv")

# 5. Save the Sparse Matrix (Transposed for R)
mmwrite(output_dir + "/matrix_hvg.mtx", adata_hvg.X.T)

In [ ]:
#plot for QC

#sc.tl.umap(adata, min_dist=1.5, spread=2) #changed these values based on previous plots
with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100)
}):
    QC_plot = sc.pl.umap(adata,
               color = ['sample', "n_genes_by_counts",
                        'phase', "integrated_celltype","celltype_fine", "celltype_coarse"],
               wspace = 0.6,
               ncols = 3,
               alpha = 0.75,
               size = 10,
               save = "_harmony_QC.pdf"
    #            legend_loc = 'on data',
              )


QC_plot

In [ ]:
#inferCNV
#inferCNV_file = '/nfs/team292/rs40/projects/scRNAseq_aneu_practice01/processed_data/inferCNV/petropoulos/ploidy_cell_petropoulos.csv'

#scploid
scploid_df = pd.read_csv( "/nfs/team292/rs40/projects/Aneuploid_screen_v2/processed_data/4_scploid/cell_karyotype_scploid.csv", index_col=0)


In [ ]:
scploid_df

In [ ]:
# adata with ploidy info
adata.obs = adata.obs.set_index('cell.ID_').join(scploid_df.set_index('cell'), how='left', sort=False)

In [ ]:
adata.obs['cellID'] = adata.obs.index

In [ ]:
adata.obs.head()

In [ ]:
adata.obs['ploidy_scploid'] = adata.obs['ploidy']

#merge with embryo ploidy info
embryo_scploid_df = pd.read_csv( '/nfs/team292/rs40/projects/Aneuploid_screen/processed_data/general_plots_celltype/ploidy_embryo.csv', index_col=0)
embryo_scploid_df['dataset'] = embryo_scploid_df['sample'] 

embryo_scploid_df = embryo_scploid_df[embryo_scploid_df['ploidy'] == 'complex']
embryo_scploid_df = embryo_scploid_df.drop(columns='ploidy').rename(columns={'percentage': 'per_complex'})[["time", "filtered_feature_call", "count",'per_complex',  "dataset"]]

embryo_scploid_df.head()

adata.obs= pd.merge(adata.obs, embryo_scploid_df, on=['dataset', 'filtered_feature_call'], how='left')

adata.obs

In [ ]:
#visualise the clusters so far
#plot for QC
with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100),
    "savefig.dpi" :(300)
}):
    g=sc.pl.umap(adata, color=["ploidy"],
           #legend_loc="on data",
           ncols = 4,
           legend_fontsize = 'xx-small',
           size = 5,
           alpha = 0.75,
           palette= aneu_col, 
           wspace=0.5,
          save = "_ploidy.pdf")

g

In [ ]:
#visualise the clusters so far
#plot for QC
with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (2.5, 2.5), 
    "figure.dpi": (100),
    "savefig.dpi" :(300)
}):
    g=sc.pl.umap(adata, color=["integrated_celltype"],
           #legend_loc="on data",
           ncols = 4,
           #legend_fontsize = 'xx-small',
           size = 4,
           alpha = 0.75,
           palette= celltype_colors, 
           wspace=0.5,
          save = "_celltype.pdf")

g

In [ ]:
adata.obs

## Specific Genes

In [ ]:
filtered_marker_genes = {
    "Naive hPSC": {"KLF17", "DPPA5", "IL6ST", "ZNF729"},
    "ICM": {"ATG2A",  "ESRRB", "LAMA4", "EPHA4"},
    "Epiblast": {"POU5F1", "NANOG", "NODAL", "KLF17"},
    "Trophectoderm": {"GATA3", "CDX2", "TFAP2C", "KRT7"},
    "Hypoblast": {"PDGFRA", "GATA6", "GATA4", "SOX17"},
    
}

In [ ]:
# plt.rcParams.update({'font.size': 4})
# plt.rcParams['ytick.labelsize'] = 5

# FIGSIZE=(3,3.5)
# #rcParams['figure.figsize']=FIGSIZE
# sc.set_figure_params(scanpy=True, fontsize=9)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
}):
    # Make Axes
    # Number of needed rows and columns (based on the row with the most columns)
    nrow=len(filtered_marker_genes)
    ncol=max([len(vs) for vs in filtered_marker_genes.values()])
    fig,axs=plt.subplots(nrow,ncol,figsize=(2.5*ncol,2*nrow))
    # Plot expression for every marker on the corresponding Axes object
    for row_idx,(cell_type,markers) in enumerate(filtered_marker_genes.items()):
        col_idx=0
        for marker in markers:
            ax=axs[row_idx,col_idx]
            sc.pl.umap(adata,color=marker, size = 10, ax=ax,show=False,frameon=False, cmap="RdPu")
            
            # Add cell type as row label - here we simply add it as ylabel of
            # the first Axes object in the row
            if col_idx==0:
                # We disabled axis drawing in UMAP to have plots without background and border
                # so we need to re-enable axis to plot the ylabel
                ax.axis('on')
                ax.set(xlabel=None)
                ax.tick_params(
                    top='off', bottom='off', left='off', right='off', 
                    labelleft='off', labelbottom='off')
                ax.set_ylabel(cell_type+'\n', rotation=90)
                ax.set(frame_on=False)
            col_idx+=1
        # Remove unused column Axes in the current row
        while col_idx<ncol:
            axs[row_idx,col_idx].remove()
            col_idx+=1

# Alignment within the Figure
fig.tight_layout()
fig.savefig(output_dir +'/'+'umaps.pdf')

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc

celltype_colors = {
    'Naive hPSC' :"#F8D56B",
    "ICM": "#DC4C5D",
    "Epiblast": "#f69320",
    "Trophectoderm": "#4f5b9e",
    "unspecified": "#676868",
     "Hypoblast": "#714684",
}

# optional: keep PDF vector/editable
sc.set_figure_params(
    vector_friendly=True,  # usage for PDF/SVG editors (Illustrator)
     # Default save format
)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False
}):
    nrow = len(filtered_marker_genes)
    ncol = max(len(vs) for vs in filtered_marker_genes.values())

    fig, axs = plt.subplots(
        nrow,
        ncol,
        figsize=(2.5 * ncol, 2 * nrow),
        squeeze=False
    )

    for row_idx, (cell_type, markers) in enumerate(filtered_marker_genes.items()):

        # make row-specific colormap
        row_color = celltype_colors[cell_type]
        row_cmap = LinearSegmentedColormap.from_list(
            f"{cell_type}_cmap",
            ["#f0f0f0", row_color]
        )

        col_idx = 0

        for marker in markers:
            ax = axs[row_idx, col_idx]

            sc.pl.umap(
                adata,
                color=marker,
                size=10,
                ax=ax,
                show=False,
                frameon=False,
                cmap=row_cmap,
                vmin=0
                #vmax="p99"
            )

            # keep dots as vector in PDF
            for coll in ax.collections:
                coll.set_rasterized(True)

            if col_idx == 0:
                ax.axis("on")
                ax.set(xlabel=None)
                ax.tick_params(
                    top=False,
                    bottom=False,
                    left=False,
                    right=False,
                    labelleft=False,
                    labelbottom=False
                )
                ax.set_ylabel(cell_type + "\n", rotation=90)
                ax.set(frame_on=False)

            col_idx += 1

        while col_idx < ncol:
            axs[row_idx, col_idx].remove()
            col_idx += 1

fig.tight_layout()
fig.savefig(output_dir + "/umaps.pdf")

In [ ]:
markers = {'celltype': ['LAMA4', 'LEF1','NANOG', 'GATA3', 'GATA6' ],
           'implantation':["BSG"],
             'immunity':['C3','CD46', 'CD55','CD276','CD47'],
          'adhesion' :['CDH1', 'ITGB1', 'ITGA6','ITGAV', 'ITGB5']}

In [ ]:
markers = {  'adhesion' :['CDH1', 'ITGB1', 'ITGA6','ITGAV', 'ITGB5']}

In [ ]:
dp = sc.pl.dotplot(
    adata, markers, groupby="celltype_coarse",
    cmap=white_to_cyan,
    swap_axes=True,          # puts genes on x-axis
    return_fig=True, show=False
)
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)

In [ ]:
# plot for Day0
dp = sc.pl.dotplot(
    adata[adata.obs["timepoint"] == "day1"], markers, groupby="ploidy_scploid",
    cmap=white_to_cyan,
    swap_axes=True,          # puts genes on x-axis
    return_fig=True, show=False
)
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)

dp.fig.set_size_inches(5, 5)            # (width, height) in inches
dp.fig.tight_layout()                   # tidy spacing

# Save (vector PDF) — crisp in papers
dp.fig.savefig(path +"/dotplot_D0_ploidy.pdf", bbox_inches="tight")

In [ ]:
sc.pl.violin(
    adata[adata.obs["timepoint"] == "day1"],
    keys="CDH1",
    groupby="ploidy_scploid",     # or "celltype_coarse"
    jitter=0.4,
    stripplot=True,
    rotation=90
)

In [ ]:
# for Day4

ploidy_order = ["euploid", "trisomy", "monosomy", "complex"]

sub = adata[adata.obs["timepoint"] == "day4"].copy()
sub.obs["panel"] = pd.Categorical(
    sub.obs["ploidy_scploid"].astype(str),
    categories=ploidy_order,
    ordered=True
)

# build combined group label: "<ploidy> | <annotation>"
sub.obs["grp"] = sub.obs["celltype_coarse"].str.cat(sub.obs["panel"].astype(str), sep=" | ")

# order rows by ploidy block, then annotation
order = (sub.obs
         .sort_values(["celltype_coarse", "panel"])
         ["grp"].drop_duplicates().tolist())
sub.obs["grp"] = pd.Categorical(sub.obs["grp"], categories=order, ordered=True)


# prevent implicit displays inside this block
# --- build the dotplot (no auto-show) ---
dp = sc.pl.dotplot(
    sub, markers, groupby="grp",
    cmap=white_to_cyan, swap_axes=True,
    return_fig=True, show=False
)

# --- edit the ACTUAL axes inside dp ---
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_title("Day4", fontsize=12)

# size & layout
dp.fig.set_size_inches(12, 5)
dp.fig.subplots_adjust(top=0.90)     # extra room for top labels


# save 
out_file = path + "/dotplot_D4_ploidy_by_annotation.pdf"
dp.fig.savefig(out_file, bbox_inches="tight")
    
# plt
display(dp.fig)
plt.close(dp.fig)

In [ ]:
# for Day6

ploidy_order = ["euploid", "trisomy", "monosomy", "complex"]

sub = adata[adata.obs["timepoint"] == "day6"].copy()
sub.obs["panel"] = pd.Categorical(
    sub.obs["ploidy_scploid"].astype(str),
    categories=ploidy_order,
    ordered=True
)

# build combined group label: "<ploidy> | <annotation>"
sub.obs["grp"] = sub.obs["celltype_coarse"].str.cat(sub.obs["panel"].astype(str), sep=" | ")

# order rows by ploidy block, then annotation
order = (sub.obs
         .sort_values(["celltype_coarse", "panel"])
         ["grp"].drop_duplicates().tolist())
sub.obs["grp"] = pd.Categorical(sub.obs["grp"], categories=order, ordered=True)


# prevent implicit displays inside this block
# --- build the dotplot (no auto-show) ---
dp = sc.pl.dotplot(
    sub, markers, groupby="grp",
    cmap=white_to_cyan, swap_axes=True,
    return_fig=True, show=False
)

# --- edit the ACTUAL axes inside dp ---
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_title("Day6", fontsize=12)

# size & layout
dp.fig.set_size_inches(12, 5)
dp.fig.subplots_adjust(top=0.90)     # extra room for top labels


# save 
out_file = path + "/dotplot_D6_ploidy_by_annotation.pdf"
dp.fig.savefig(out_file, bbox_inches="tight")
    
# plt
display(dp.fig)
plt.close(dp.fig)

In [ ]:
#apoptotic
markers = {'core': ["CASP3","CASP7","DFFA","DFFB"],
           'TP53':["PMAIP1","BBC3","TP53"],
             'NF-κB':['C3','CD46', 'CD55','CD276','CD47'],
          'death receptor' :["FAS","FASLG","TNFSF10","TNFRSF10A","TNFRSF10B","FADD","CASP8","CASP10","CFLAR","BID"],
          'Phagocytic-opsonin':["MFGE8","GAS6","PROS1","C1QA","C1QC","C3","THBS1","PTX3"],
          'PtdSer' :["XKR8","XKR4","XKR9","ANO6","ATP11C","ATP11A","TMEM30A","CALR","ANXA1"]}

In [ ]:
dp = sc.pl.dotplot(
    adata, markers, groupby="celltype_coarse",
    cmap=white_to_magenta,
    swap_axes=True,          # puts genes on x-axis
    return_fig=True, show=False
)
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)

In [ ]:
# plot for Day0
dp = sc.pl.dotplot(
    adata[adata.obs["timepoint"] == "day1"], markers, groupby="ploidy_scploid",
    cmap=white_to_magenta,
    swap_axes=True,          # puts genes on x-axis
    return_fig=True, show=False
)
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)

dp.fig.set_size_inches(5, 10)            # (width, height) in inches
dp.fig.tight_layout()                   # tidy spacing

# Save (vector PDF) — crisp in papers
dp.fig.savefig(path +"/dotplot_D0_ploidy_apoptosis.pdf", bbox_inches="tight")

In [ ]:
# for Day4

ploidy_order = ["euploid", "trisomy", "monosomy", "complex"]

sub = adata[adata.obs["timepoint"] == "day4"].copy()
sub.obs["panel"] = pd.Categorical(
    sub.obs["ploidy_scploid"].astype(str),
    categories=ploidy_order,
    ordered=True
)

# build combined group label: "<ploidy> | <annotation>"
sub.obs["grp"] = sub.obs["celltype_coarse"].str.cat(sub.obs["panel"].astype(str), sep=" | ")

# order rows by ploidy block, then annotation
order = (sub.obs
         .sort_values(["celltype_coarse", "panel"])
         ["grp"].drop_duplicates().tolist())
sub.obs["grp"] = pd.Categorical(sub.obs["grp"], categories=order, ordered=True)


# prevent implicit displays inside this block
# --- build the dotplot (no auto-show) ---
dp = sc.pl.dotplot(
    sub, markers, groupby="grp",
    cmap=white_to_magenta, swap_axes=True,
    return_fig=True, show=False
)

# --- edit the ACTUAL axes inside dp ---
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_title("Day4", fontsize=12)

# size & layout
dp.fig.set_size_inches(12, 10)
dp.fig.subplots_adjust(top=0.90)     # extra room for top labels


# save 
out_file = path + "/dotplot_D4_apoptosis.pdf"
dp.fig.savefig(out_file, bbox_inches="tight")
    
# plt
display(dp.fig)
plt.close(dp.fig)

In [ ]:
# for Day6

ploidy_order = ["euploid", "trisomy", "monosomy", "complex"]

sub = adata[adata.obs["timepoint"] == "day6"].copy()
sub.obs["panel"] = pd.Categorical(
    sub.obs["ploidy_scploid"].astype(str),
    categories=ploidy_order,
    ordered=True
)

# build combined group label: "<ploidy> | <annotation>"
sub.obs["grp"] = sub.obs["celltype_coarse"].str.cat(sub.obs["panel"].astype(str), sep=" | ")

# order rows by ploidy block, then annotation
order = (sub.obs
         .sort_values(["celltype_coarse", "panel"])
         ["grp"].drop_duplicates().tolist())
sub.obs["grp"] = pd.Categorical(sub.obs["grp"], categories=order, ordered=True)


# prevent implicit displays inside this block
# --- build the dotplot (no auto-show) ---
dp = sc.pl.dotplot(
    sub, markers, groupby="grp",
    cmap=white_to_magenta, swap_axes=True,
    return_fig=True, show=False
)

# --- edit the ACTUAL axes inside dp ---
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_title("Day6", fontsize=12)

# size & layout
dp.fig.set_size_inches(12, 10)
dp.fig.subplots_adjust(top=0.90)     # extra room for top labels


# save 
out_file = path + "/dotplot_D6_apoptosis.pdf"
dp.fig.savefig(out_file, bbox_inches="tight")
    
# plt
display(dp.fig)
plt.close(dp.fig)

## Differential expression

In [ ]:
adata.uns['log1p']["base"] = None

In [ ]:
adata.obs['ploidy_scploid']

In [ ]:
adata.obs['ploidy_scploid'] = adata.obs['ploidy_scploid'].cat.reorder_categories(['diploid', 'trisomy','monosomy','complex'])

In [ ]:
adata.obs["celltype_coarse"] = adata.obs["celltype_coarse"].astype(str)

In [ ]:
adata.obs['celltype_coarse'].unique()

In [ ]:
adata.obs["celltype_category"] = adata.obs['celltype_coarse'].apply(
    lambda x: 
    "TE" if "TE" in x or "STB" in x else 
    "EPI" if "EPI" in x 
    else x
)

In [ ]:
adata.obs["celltype_category"].unique()

In [ ]:
# By ploidy type

for cell_type in adata.obs["celltype_category"].unique() :
    adata_sub = adata[adata.obs["celltype_category"] == cell_type]
    sc.tl.rank_genes_groups(adata_sub, groupby = "ploidy_scploid")
    #sc.pl.rank_genes_groups_heatmap(adata_sub, groupby="ploidy_scploid", n_genes=5, save="_ploidy_"+cell_type, dendrogram=False)
    sc.pl.rank_genes_groups_matrixplot(adata_sub, groupby="ploidy_scploid", n_genes=5, save="_ploidy_"+cell_type, 
                                       dendrogram=False, 
                                       title = cell_type, 
                                       #swap_axes = True, 
                                       vmax = 2.5

)

## Pathway activity scoring

An alternative approach is to simply score the activity of a pathway or gene signature, in absolute sense, in individual cells, rather than testing for a differential activity between conditions.

### mlm on PROGENy

PROGENy is a comprehensive resource containing a curated collection of pathways and their target genes, with weights for each interaction. For this example we will use the human weights (other organisms are available) and we will use the top 500 responsive genes ranked by p-value. Here is a brief description of each pathway:

In [ ]:
progeny = dc.op.progeny(organism="human", top=500)
progeny

In [ ]:
dc.mt.mlm(data=adata, net=progeny)
score = dc.pp.get_obsm(adata=adata, key="score_mlm")

In [ ]:
# scores: cells x pathways
scores_df = score.to_df()
scores_df.index.name = "cell"
scores_df.to_csv(output_dir + "/progeny_mlm_scores.csv")

In [ ]:
scores_df.head()

In [ ]:
sc.pl.matrixplot(score, var_names=score.var_names, groupby=['celltype_coarse'], dendrogram=True, standard_scale='var',
                 colorbar_title='Z-scaled scores', cmap='YlGnBu' )

In [ ]:
#day0
hp = sc.pl.matrixplot(score[score.obs["timepoint"] == "day1"], 
                 var_names=score.var_names, 
                 groupby='ploidy_scploid', 
                 dendrogram=False, standard_scale='var',
                 colorbar_title='Z-scaled scores',
                 cmap='YlGnBu' ,swap_axes=True, return_fig=True, show=False)

ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_title("Day0", fontsize=12)

hp.fig.set_size_inches(3, 5)            # (width, height) in inches
hp.fig.tight_layout()                   # tidy spacing


# Save (vector PDF) — crisp in papers
hp.fig.savefig(path +"/pathway_D0_ploidy.pdf")

In [ ]:
# for Day4
ploidy_order = ["euploid", "trisomy", "monosomy", "complex"]

sub = score[score.obs["timepoint"] == "day4"].copy()

sub.obs["panel"] = pd.Categorical(
    sub.obs["ploidy_scploid"].astype(str),
    categories=ploidy_order,
    ordered=True
)

# build combined group label: "<ploidy> | <annotation>"
sub.obs["grp"] = sub.obs["celltype_coarse"].str.cat(sub.obs["panel"].astype(str), sep=" | ")

# order rows by ploidy block, then annotation
order = (sub.obs
         .sort_values(["celltype_coarse", "panel"])
         ["grp"].drop_duplicates().tolist())
sub.obs["grp"] = pd.Categorical(sub.obs["grp"], categories=order, ordered=True)


# prevent implicit displays inside this block
# --- build the dotplot (no auto-show) ---
hp = sc.pl.matrixplot(sub, 
                 var_names=score.var_names, 
                 groupby='grp', 
                 dendrogram=False, standard_scale='var',
                 colorbar_title='Z-scaled scores',
                 cmap='YlGnBu' ,swap_axes=True, return_fig=True, show=False)

# --- edit the ACTUAL axes inside dp ---
ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_title("Day4", fontsize=12)

# size & layout
hp.fig.set_size_inches(9, 5)
hp.fig.subplots_adjust(top=0.90)     # extra room for top labels


# save 
out_file = path + "/pathway_D4_progeny.pdf"
hp.fig.savefig(out_file, bbox_inches="tight")
    
# plt
display(hp.fig)
plt.close(hp.fig)

In [ ]:
# for Day6
ploidy_order = ["euploid", "trisomy", "monosomy", "complex"]

sub = score[score.obs["timepoint"] == "day6"].copy()

sub.obs["panel"] = pd.Categorical(
    sub.obs["ploidy_scploid"].astype(str),
    categories=ploidy_order,
    ordered=True
)

# build combined group label: "<ploidy> | <annotation>"
sub.obs["grp"] = sub.obs["celltype_coarse"].str.cat(sub.obs["panel"].astype(str), sep=" | ")

# order rows by ploidy block, then annotation
order = (sub.obs
         .sort_values(["celltype_coarse", "panel"])
         ["grp"].drop_duplicates().tolist())
sub.obs["grp"] = pd.Categorical(sub.obs["grp"], categories=order, ordered=True)


# prevent implicit displays inside this block
# --- build the dotplot (no auto-show) ---
hp = sc.pl.matrixplot(sub, 
                 var_names=score.var_names, 
                 groupby='grp', 
                 dendrogram=False, standard_scale='var',
                 colorbar_title='Z-scaled scores',
                 cmap='YlGnBu' ,swap_axes=True, return_fig=True, show=False)

# --- edit the ACTUAL axes inside dp ---
ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_title("Day6", fontsize=12)

# size & layout
hp.fig.set_size_inches(9, 5)
hp.fig.subplots_adjust(top=0.90)     # extra room for top labels


# save 
out_file = path + "/pathway_D6_progeny.pdf"
hp.fig.savefig(out_file, bbox_inches="tight")
    
# plt
display(hp.fig)
plt.close(hp.fig)

### mlm on Hallmark

https://decoupler.readthedocs.io/en/latest/notebooks/scell/rna_sc.html#hallmark-gene-sets

Hallmark gene sets are curated collections of genes that represent specific, well-defined biological states or processes. They are part of MSigDB and were developed to reduce redundancy and improve interpretability compared to older, more overlapping gene set collections

In [ ]:
hallmark = dc.op.hallmark(organism="human")
hallmark

In [ ]:
dc.mt.mlm(data=adata, net=hallmark)
score = dc.pp.get_obsm(adata=adata, key="score_mlm")
score

In [ ]:
# scores: cells x pathways
scores_df = score.to_df()
scores_df.index.name = "cell"
scores_df.to_csv(output_dir + "/hallmark_mlm_scores.csv")

In [ ]:
score.var_names

In [ ]:
hallmarks_sub = ['APICAL_JUNCTION', 'APICAL_SURFACE', 'APOPTOSIS',
       'COMPLEMENT', 'DNA_REPAIR', 'E2F_TARGETS',
       'EPITHELIAL_MESENCHYMAL_TRANSITION', 'ESTROGEN_RESPONSE_EARLY',
       'ESTROGEN_RESPONSE_LATE', 'FATTY_ACID_METABOLISM', 'G2M_CHECKPOINT',
       'GLYCOLYSIS', 'HEDGEHOG_SIGNALING', 'HYPOXIA',
       'IL2_STAT5_SIGNALING', 'IL6_JAK_STAT3_SIGNALING',
       'INFLAMMATORY_RESPONSE', 'INTERFERON_ALPHA_RESPONSE',
       'INTERFERON_GAMMA_RESPONSE', 'KRAS_SIGNALING_DN', 'KRAS_SIGNALING_UP',
       'MITOTIC_SPINDLE', 'MTORC1_SIGNALING', 'MYC_TARGETS_V1',
       'MYC_TARGETS_V2', 'NOTCH_SIGNALING',
       'OXIDATIVE_PHOSPHORYLATION', 'P53_PATHWAY', 
       'PEROXISOME', 'PI3K_AKT_MTOR_SIGNALING', 'PROTEIN_SECRETION',
       'REACTIVE_OXYGEN_SPECIES_PATHWAY', 
       'TGF_BETA_SIGNALING', 'TNFA_SIGNALING_VIA_NFKB',
       'UNFOLDED_PROTEIN_RESPONSE', 'UV_RESPONSE_DN',
       'WNT_BETA_CATENIN_SIGNALING']

In [ ]:
sc.pl.matrixplot(
    adata=score,
    var_names=hallmarks_sub,
    groupby='celltype_coarse',
    dendrogram=True,
    standard_scale="var",
    colorbar_title="Z-scaled scores",
    cmap="YlOrRd"
)


In [ ]:
#day0
hp = sc.pl.matrixplot(score[score.obs["timepoint"] == "day1"], 
                 var_names=hallmarks_sub, 
                 groupby='ploidy_scploid', 
                 dendrogram=False, 
                 standard_scale='var',
                 colorbar_title='Z-scaled scores',
                 cmap='RdPu' ,swap_axes=True, return_fig=True, show=False)

ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_title("Day0", fontsize=12)

hp.fig.set_size_inches(8, 8)            # (width, height) in inches
hp.fig.tight_layout()                   # tidy spacing


# Save (vector PDF) — crisp in papers
hp.fig.savefig(path +"/pathway_D0_hallmark.pdf")

In [ ]:
# for Day4
ploidy_order = ["euploid", "trisomy", "monosomy", "complex"]

sub = score[score.obs["timepoint"] == "day4"].copy()

sub.obs["panel"] = pd.Categorical(
    sub.obs["ploidy_scploid"].astype(str),
    categories=ploidy_order,
    ordered=True
)

# build combined group label: "<ploidy> | <annotation>"
sub.obs["grp"] = sub.obs["celltype_coarse"].str.cat(sub.obs["panel"].astype(str), sep=" | ")

# order rows by ploidy block, then annotation
order = (sub.obs
         .sort_values(["celltype_coarse", "panel"])
         ["grp"].drop_duplicates().tolist())
sub.obs["grp"] = pd.Categorical(sub.obs["grp"], categories=order, ordered=True)


# prevent implicit displays inside this block
# --- build the dotplot (no auto-show) ---
hp = sc.pl.matrixplot(sub, 
                 var_names=hallmarks_sub, 
                 groupby='grp', 
                 dendrogram=False, standard_scale='var',
                 colorbar_title='Z-scaled scores',
                 cmap='RdPu' ,swap_axes=True, return_fig=True, show=False)

# --- edit the ACTUAL axes inside dp ---
ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_title("Day4", fontsize=12)

# size & layout
hp.fig.set_size_inches(9, 8)
hp.fig.subplots_adjust(top=0.90)     # extra room for top labels


# save 
out_file = path + "/pathway_D4_hallmark.pdf"
hp.fig.savefig(out_file, bbox_inches="tight")
    
# plt
display(hp.fig)
plt.close(hp.fig)

## TF activity

In [ ]:
net = dc.get_collectri(organism='human', split_complexes=False)
net

In [ ]:
dc.run_ulm(
    mat=adata,
    net=net,
    source='source',
    target='target',
    weight='weight',
    use_raw=False
)

In [ ]:
acts = dc.get_acts(adata, obsm_key='ulm_estimate')

In [ ]:
df = dc.rank_sources_groups(acts, groupby='celltype_coarse', reference='rest', method='t-test_overestim_var')

n_markers = 5
source_markers = df.groupby('group').head(n_markers).groupby('group')['names'].apply(lambda x: list(x)).to_dict()
source_markers

sc.pl.matrixplot(acts, source_markers, 'celltype_coarse', dendrogram=True, standard_scale='var',
                 colorbar_title='Z-scaled scores', cmap='RdBu_r')

## For cellphoneDB

I will try to find out
1. differentially expressed genes for euploid and aneuploid cells(complex) in each cell type
2. differentially expressed genes for euploid in mosaic and aneuploid in mosaic. 

### for celltypes

In [ ]:
#make a df for DEG for different aneuploid cells in EPI cells
#groups=['complex', 'euploid']
adata.obs['celltype_coarse'].unique()
adata_sub = adata.copy()

In [ ]:
min_count =  adata_sub.obs['celltype_coarse'].value_counts().min()

# Randomly sample min_count cells from each group
balanced_indices = (
    adata_sub.obs
    .groupby('celltype_coarse', group_keys=False, observed=False)
    .apply(lambda x: x.sample(min_count, random_state=0))
    .index
)

# Subset the AnnData object
adata_sub = adata_sub[balanced_indices].copy()

In [ ]:
adata_sub.obs['celltype_coarse'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(adata_sub,
                        #reference = "euploid", 
                        #groups=groups, 
                        groupby = "celltype_coarse")


deg_df_dict = {}

for cell_type in adata_sub.obs['celltype_coarse'].unique() :
    #sc.pl.rank_genes_groups_heatmap(adata_sub, groupby="ploidy_scploid", n_genes=5, save="_ploidy_"+cell_type, dendrogram=False)
    deg_df_dict[cell_type] = sc.get.rank_genes_groups_df(adata_sub, group = cell_type)
    deg_df_dict[cell_type]  = deg_df_dict[cell_type][(deg_df_dict[cell_type] ['pvals_adj'] < 0.05) & (deg_df_dict[cell_type]['logfoldchanges'] > 0.25)]
    deg_df_dict[cell_type]['cell_type'] = cell_type
    

DEG_df = pd.concat(list(deg_df_dict.values()))

DEG_df

In [ ]:
#reformat and save for cell phone DB
DEG_df_cpdb = pd.concat([DEG_df['cell_type'], DEG_df['names']], axis = 1)

DEG_df_cpdb= DEG_df_cpdb.rename(columns={"names": "genes"})
DEG_df_cpdb['cell_type']= DEG_df_cpdb['cell_type'].astype('string')

In [ ]:
DEG_df_cpdb.to_csv(output_dir + '/deg.tsv', sep='\t', index=False)
adata_sub.obs.to_csv(output_dir +'/cpdb_meta.csv')
adata_sub.obs.index = adata_sub.obs['cellID']

In [ ]:
output_dir + '/deg.tsv'

In [ ]:
adata

In [ ]:
del adata_sub._obsm['ora_estimate']
del adata_sub._obsm['ora_pvals']

In [ ]:
adata_sub.write_h5ad(output_dir+"/adata_cpdb" +".h5ad")

In [ ]:
adata_sub.obs

In [ ]:
sc.pl.dotplot(adata,["EFNA1", "EFNA3",  "EFNA4",  "EFNA5",
                     "EPHA1", "EPHA3", "EPHA4","EPHA5", "EPHA7"], groupby=['celltype_coarse', 'ploidy_scploid'], save = "_ephrins.pdf")

### for complex and euploid cells of EPI

In [ ]:
#make a df for DEG for different aneuploid cells in EPI cells
#groups=['complex', 'euploid']

adata_sub = adata[
    (adata.obs['celltype_category'] == "unknown") &
    (adata.obs['dataset'].isin(["T2_control", "T2_mix", "T2_rev"]))]
adata_sub.obs['ploidy_scploid'].unique()

In [ ]:
min_count =  adata_sub.obs['ploidy_scploid'].value_counts().min()

# Randomly sample min_count cells from each group
balanced_indices = (
    adata_sub.obs
    .groupby('ploidy_scploid', group_keys=False, observed=False)
    .apply(lambda x: x.sample(min_count, random_state=0))
    .index
)

# Subset the AnnData object
adata_sub = adata_sub[balanced_indices].copy()

In [ ]:
adata_sub.obs['ploidy_scploid'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(adata_sub,
                        #reference = "euploid", 
                        #groups=groups, 
                        groupby = "ploidy_scploid")


deg_df_dict = {}

for cell_type in adata_sub.obs['ploidy_scploid'].unique() :
    #sc.pl.rank_genes_groups_heatmap(adata_sub, groupby="ploidy_scploid", n_genes=5, save="_ploidy_"+cell_type, dendrogram=False)
    deg_df_dict[cell_type] = sc.get.rank_genes_groups_df(adata_sub, group = cell_type)
    deg_df_dict[cell_type]  = deg_df_dict[cell_type][(deg_df_dict[cell_type] ['pvals_adj'] < 0.05) & (deg_df_dict[cell_type]['logfoldchanges'] > 0.25)]
    deg_df_dict[cell_type]['cell_type'] = cell_type
    

DEG_df = pd.concat(list(deg_df_dict.values()))

DEG_df

In [ ]:
#reformat and save for cell phone DB
DEG_df_cpdb = pd.concat([DEG_df['cell_type'], DEG_df['names']], axis = 1)

DEG_df_cpdb= DEG_df_cpdb.rename(columns={"names": "genes"})


In [ ]:
DEG_df_cpdb['cell_type']= DEG_df_cpdb['cell_type'].astype('string')

In [ ]:
DEG_df_cpdb['cell_type'].dtype

In [ ]:
DEG_df_cpdb.to_csv(output_dir + '/deg.tsv', sep='\t', index=False)

In [ ]:
output_dir + '/deg.tsv'

In [ ]:
adata_sub.obs.to_csv(output_dir +'/cpdb_meta.csv')

In [ ]:
adata_sub.obs.index = adata_sub.obs['cellID']

adata_sub.write_h5ad(output_dir+"/adata_cpdb" +".h5ad")

In [ ]:
adata_sub.obs

In [ ]:
sc.pl.dotplot(adata,["EFNA1", "EFNA3",  "EFNA4",  "EFNA5",
                     "EPHA1", "EPHA3", "EPHA4","EPHA5", "EPHA7"], groupby=['celltype_coarse', 'ploidy_scploid'], save = "_ephrins.pdf")

## Further group based on ploidy percent

In [ ]:
adata.obs

In [ ]:
#make a df for DEG for different aneuploid cells in EPI cells
#groups=['complex', 'euploid']

adata_sub = adata[
    (adata.obs['celltype_category'] == "unknown") &
    (adata.obs['dataset'].isin(["T2_control", "T2_mix", "T2_rev"]))&
     (~adata.obs['complexity'].isna())]
adata_sub.obs['ploidy_scploid'].unique()

In [ ]:
adata_sub.obs['complexity'].unique()

In [ ]:
adata_sub.obs['ploidy_proportion'] = adata_sub.obs['ploidy_scploid'].astype(str) +"_"+  adata_sub.obs['complexity'].astype(str)

In [ ]:
min_count =  adata_sub.obs['ploidy_proportion'].value_counts().min()

# Randomly sample min_count cells from each group
balanced_indices = (
    adata_sub.obs
    .groupby('ploidy_proportion', group_keys=False, observed=False)
    .apply(lambda x: x.sample(min_count, random_state=0))
    .index
)

# Subset the AnnData object
adata_sub = adata_sub[balanced_indices].copy()

In [ ]:
adata_sub.obs['ploidy_proportion'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(adata_sub,
                        #reference = "euploid", 
                        #groups=groups, 
                        groupby = "ploidy_proportion")


deg_df_dict = {}

for cell_type in adata_sub.obs['ploidy_proportion'].unique() :
    #sc.pl.rank_genes_groups_heatmap(adata_sub, groupby="ploidy_scploid", n_genes=5, save="_ploidy_"+cell_type, dendrogram=False)
    deg_df_dict[cell_type] = sc.get.rank_genes_groups_df(adata_sub, group = cell_type)
    deg_df_dict[cell_type]  = deg_df_dict[cell_type][(deg_df_dict[cell_type] ['pvals_adj'] < 0.05) & (deg_df_dict[cell_type]['logfoldchanges'] > 0.25)]
    deg_df_dict[cell_type]['cell_type'] = cell_type
    

DEG_df = pd.concat(list(deg_df_dict.values()))

DEG_df

In [ ]:
#reformat and save for cell phone DB
DEG_df_cpdb = pd.concat([DEG_df['cell_type'], DEG_df['names']], axis = 1)

DEG_df_cpdb= DEG_df_cpdb.rename(columns={"names": "genes"})


In [ ]:
DEG_df_cpdb['cell_type']= DEG_df_cpdb['cell_type'].astype('string')

In [ ]:
DEG_df_cpdb['cell_type'].dtype

In [ ]:
DEG_df_cpdb.to_csv(output_dir + '/deg.tsv', sep='\t', index=False)

In [ ]:
output_dir + '/deg.tsv'

In [ ]:
adata_sub.obs.to_csv(output_dir +'/cpdb_meta.csv')

In [ ]:
adata_sub.obs.index = adata_sub.obs['cellID']

In [ ]:
del adata_sub._obsm['ora_estimate']
del adata_sub._obsm['ora_pvals']
adata_sub.write_h5ad(output_dir+"/adata_cpdb" +".h5ad")

## Gene set tests

Gene set tests test whether a pathway is enriched, in other words over-represented, in one condition compared to others, say, in healthy donors compared to severe COVID-19 patients in the monocyte population

### Over Representation Analysis

using ORA: a statistical method used to identify pathways or gene sets that are significantly enriched in a subset of genes — for example, those highly expressed or differentially expressed in your experiment.
Compares the DEGs from your analysis with predefined gene sets (pathways) to determine if any of these gene sets are disproportionately represented compared to what would be expected by chance. Statistical tests are used to assess whether the overlap between your DEGs and pathway genes is significant. (which pathway is over represented in DEG?)

In [ ]:
adata

In [ ]:
msigdb_original = dc.op.resource("MSigDB")

In [ ]:
msigdb_original['collection'].unique()

In [ ]:
# Filter by hallmark
msigdb = msigdb_original[msigdb_original['collection']=='hallmark'].copy()

# Remove duplicated entries
msigdb = msigdb[~msigdb.duplicated(['geneset', 'genesymbol'])]
msigdb

In [ ]:
# msigdb likely has columns: geneset, genesymbol
net = msigdb.rename(columns={"geneset": "source", "genesymbol": "target"})

# choose background
n_bg = adata.n_vars  # e.g. 33540
top_k = int(np.ceil(0.05 * n_bg))   # top 5%
n_up = n_bg - top_k                 # rank threshold so selected ~= top_k

dc.mt.ora(
    data=adata,
    net=net,
    tmin=5,
    raw=False,
    n_bg=n_bg,
    n_up=n_up,
    n_bm=0,
    verbose=True,
)

In [ ]:
acts = dc.get_acts(adata, obsm_key='ora_estimate')

# We need to remove inf and set them to the maximum value observed
acts_v = acts.X.ravel()
max_e = np.nanmax(acts_v[np.isfinite(acts_v)])
acts.X[~np.isfinite(acts.X)] = max_e

acts

In [ ]:
df = dc.rank_sources_groups(acts,groupby='celltype_coarse') 
n_markers = 4
source_markers =df.groupby('group').head(n_markers).groupby('group')['names'].apply(lambda x: list(x)).to_dict()

In [ ]:
sc.pl.matrixplot(acts, source_markers, 'celltype_coarse',
                 standard_scale='var',
                 colorbar_title='Z-scaled scores', cmap='RdBu_r', dendrogram=True,  save = "_hallmark_ploidy.pdf")

In [ ]:
df = dc.rank_sources_groups(acts, reference = "euploid", groupby='ploidy_scploid') 
n_markers = 4
source_markers =df.groupby('group').head(n_markers).groupby('group')['names'].apply(lambda x: list(x)).to_dict()


sc.pl.matrixplot(acts, source_markers, 'ploidy_scploid',
                 standard_scale='var',
                 colorbar_title='Z-scaled scores', cmap='RdBu_r', save = "_hallmark_ploidy.pdf")

In [ ]:
for cell_type in adata.obs['celltype_coarse'].unique() :
    
    adata_sub = adata[adata.obs['celltype_coarse'] == cell_type]
    
    dc.run_ora(
    mat=adata_sub,
    net=msigdb,
    source='geneset',
    target='genesymbol',
    verbose=True,
    use_raw=False
)
    acts = dc.get_acts(adata_sub, obsm_key='ora_estimate')

    # We need to remove inf and set them to the maximum value observed
    acts_v = acts.X.ravel()
    max_e = np.nanmax(acts_v[np.isfinite(acts_v)])
    
    acts.X[~np.isfinite(acts.X)] = max_e
    df = dc.rank_sources_groups(acts, reference = "euploid", groupby='ploidy_scploid') 
    n_markers = 5
    source_markers =df.groupby('group').head(n_markers).groupby('group')['names'].apply(lambda x: list(x)).to_dict()

    sc.pl.matrixplot(acts, source_markers, 'ploidy_scploid',
     standard_scale='var',
     title = cell_type, 
     show=False,
     colorbar_title='Z-scaled scores', cmap='RdBu_r', save = cell_type + "_hallmark_ploidy.pdf")

#### Further group based on ploidy percent

In [ ]:
adata.obs.tail()

In [ ]:
 plt.hist(adata.obs['per_complex'], bins=10, edgecolor='black')

In [ ]:
adata.obs['complexity'] = np.nan  # start with all NaNs
adata.obs.loc[adata.obs['per_complex'] < 50, 'complexity'] = 'low'
adata.obs.loc[adata.obs['per_complex'] >= 50, 'complexity'] = 'high'

In [ ]:
adata.obs['complexity'] = np.nan  # start with all NaNs
adata.obs.loc[adata.obs['per_complex'] <= 20, 'complexity'] = 'low'
adata.obs.loc[(adata.obs['per_complex'] > 20) & (adata.obs['per_complex'] < 60), 'complexity'] = 'medium'
adata.obs.loc[adata.obs['per_complex'] >= 60, 'complexity'] = 'high'

In [ ]:
adata_sub = adata[adata.obs['celltype_coarse'] == "unknown"]
adata_sub = adata_sub[adata_sub.obs['ploidy_scploid'] == "complex"]
adata_sub = adata_sub[~adata_sub.obs['complexity'].isna()].copy()
    
dc.run_ora(
mat=adata_sub,
net=msigdb,
source='geneset',
target='genesymbol',
verbose=True,
use_raw=False
)
acts = dc.get_acts(adata_sub, obsm_key='ora_estimate')

# We need to remove inf and set them to the maximum value observed
acts_v = acts.X.ravel()
max_e = np.nanmax(acts_v[np.isfinite(acts_v)])

acts.X[~np.isfinite(acts.X)] = max_e

In [ ]:
df = dc.rank_sources_groups(acts, groupby='complexity') 
n_markers = 5
source_markers =df.groupby('group').head(n_markers).groupby('group')['names'].apply(lambda x: list(x)).to_dict()

In [ ]:
sc.pl.matrixplot(acts, source_markers, 'complexity',
 standard_scale='var',
 show=True,
 colorbar_title='Z-scaled scores', cmap='RdBu_r', save =  "_hallmark_ploidy.pdf")

### GSEA

GSEA aggregates the per gene statistics across genes within a gene set, therefore making it possible to detect situations where all genes in a predefined set change in a small but coordinated way.  GSEA looks at the overall distribution of pathway genes within the ranked list to assess whether those pathways are more active or suppressed in your data. Are entire biological pathways are activated or repressed?